# Aula 08 — Desafios: robustez e delegação

**Aprender com as falhas e pedir ajuda com critério.**

Partimos do atendente da Aula 7 (consultar produto, frete, total) e acrescentamos:

| Tema | O que muda |
|---|---|
| Retornos claros | toda ferramenta devolve `{'ok': True, 'dados': ...}` ou `{'ok': False, 'erro': ...}` |
| Entrada conferida | `True` não vira 1 item; quantidade negativa é recusada |
| Retry com critério | só para falha temporária (timeout) e operação segura; até 3 tentativas |
| Quatro limites | tempo de requisição, tentativas, rodadas do modelo, execuções de ferramenta |
| Delegação | um coordenador pode pedir **um** parecer a um especialista (outro modelo) |

| Desafio | Tema |
|---|---|
| 1 | Entrada inválida: corrigir, não repetir |
| 2 | Falha temporária: retry e parada honesta |
| 3 | Limites de rodadas e de lote |
| 4 | Erros do serviço: 401/403, 404, 429, timeout |
| 5 | Especialista: duas camisetas ou três canecas? |
| 6 | Quando delegar (e quanto custa) |
| 7 | Miniprojeto: agente robusto com delegação |

**Antes de tudo:** rode a seção *Preparação*, com o segredo `OLLAMA_API_KEY` configurado.

---
# Preparação

In [ ]:
%pip install -q ollama

In [ ]:
import os

try:
    from google.colab import userdata
    OLLAMA_API_KEY = userdata.get('OLLAMA_API_KEY')
except ImportError:
    OLLAMA_API_KEY = os.environ.get('OLLAMA_API_KEY')
    if not OLLAMA_API_KEY:
        from getpass import getpass
        OLLAMA_API_KEY = getpass('OLLAMA_API_KEY: ')

In [ ]:
import json
import math
import re
import time

import httpx
from ollama import Client, ResponseError

HOST = 'https://ollama.com'
HEADERS = {'Authorization': f'Bearer {OLLAMA_API_KEY}'}

TIMEOUT_REQUISICAO = 60
MAX_TENTATIVAS = 3
MAX_RODADAS = 6
MAX_CHAMADAS_POR_RODADA = 4
MAX_CHAMADAS_TOTAL = 10
MAX_DELEGACOES = 1

cliente = Client(host=HOST, headers=HEADERS, timeout=TIMEOUT_REQUISICAO)

MODELO_COORDENADOR = 'gpt-oss:20b'
MODELO_ESPECIALISTA = 'gpt-oss:120b'

Confira nomes e acesso **antes** de usar dois modelos diferentes:

In [ ]:
try:
    nomes = [m.model for m in cliente.list().models]
    print(f'{len(nomes)} modelos listados')
    for m in (MODELO_COORDENADOR, MODELO_ESPECIALISTA):
        print(f'  {m}:', 'listado' if m in nomes else 'NÃO listado — troque o nome')
except ResponseError as e:
    print('Erro ao listar modelos:', e.status_code, e.error)

## Ferramentas com entrada conferida e retornos que não se confundem

Cada tipo de erro leva a uma decisão diferente:

| Erro | Exemplo | Decisão |
|---|---|---|
| `EntradaInvalida` | quantidade `-2` ou `True` | **corrigir** o dado — repetir não ajuda |
| `NaoEncontrado` | item `bermuda` | informar as opções |
| `TimeoutError` | estoque não respondeu | **repetir** (consulta é segura), até 3 vezes |

`consultar_estoque` simula um serviço instável: `FALHAS_SIMULADAS = n` faz as próximas `n` chamadas darem timeout.

In [ ]:
CATALOGO = {'camiseta': 35, 'caneca': 22}
FRETES = {'centro': 8, 'alvorada': 12}
ESTOQUE = {'camiseta': 8, 'caneca': 15}

FALHAS_SIMULADAS = 0


class EntradaInvalida(ValueError):
    '''Dado errado: não adianta repetir, é preciso corrigir.'''


class NaoEncontrado(LookupError):
    def __init__(self, mensagem, opcoes):
        super().__init__(mensagem)
        self.opcoes = opcoes


def _texto(valor, nome):
    if type(valor) is not str or not valor.strip():
        raise EntradaInvalida(f'{nome} deve ser um texto não vazio')
    return valor.strip().lower()


def _inteiro(valor, nome, minimo, maximo):
    if type(valor) is not int:
        raise EntradaInvalida(f'{nome} deve ser um número inteiro')
    if not minimo <= valor <= maximo:
        raise EntradaInvalida(f'{nome} deve estar entre {minimo} e {maximo}')
    return valor


def _valor(valor, nome, maximo):
    if type(valor) not in (int, float) or not math.isfinite(valor):
        raise EntradaInvalida(f'{nome} deve ser um número finito')
    if not 0 <= valor <= maximo:
        raise EntradaInvalida(f'{nome} deve estar entre 0 e {maximo}')
    return float(valor)


def consultar_produto(item):
    nome = _texto(item, 'item')
    if nome not in CATALOGO:
        raise NaoEncontrado(f'item {nome!r} não está no catálogo', sorted(CATALOGO))
    return {'item': nome, 'preco_unitario_reais': CATALOGO[nome]}


def consultar_frete(bairro):
    nome = _texto(bairro, 'bairro')
    if nome not in FRETES:
        raise NaoEncontrado(f'bairro {nome!r} não é atendido', sorted(FRETES))
    return {'bairro': nome, 'frete_por_pedido_reais': FRETES[nome]}


def consultar_estoque(item):
    global FALHAS_SIMULADAS
    nome = _texto(item, 'item')
    if FALHAS_SIMULADAS > 0:
        FALHAS_SIMULADAS -= 1
        raise TimeoutError('o serviço de estoque não respondeu a tempo')
    if nome not in ESTOQUE:
        raise NaoEncontrado(f'item {nome!r} sem registro de estoque', sorted(ESTOQUE))
    return {'item': nome, 'unidades_em_estoque': ESTOQUE[nome]}


def calcular_total(quantidade, preco, frete):
    quantidade = _inteiro(quantidade, 'quantidade', 1, 100)
    preco = _valor(preco, 'preco', 10_000)
    frete = _valor(frete, 'frete', 1_000)
    return {'quantidade': quantidade, 'preco_unitario_reais': preco,
            'frete_reais': frete, 'total_reais': round(quantidade * preco + frete, 2)}

## Retry só quando ajuda e é seguro

Até **três tentativas no total**, com pausas de 0,5 s e 1 s. Só para `TimeoutError` e só nas **consultas**
(ler preço/frete/estoque duas vezes não causa dano).
Uma compra ou envio só poderia ser repetido com controle de duplicatas — um timeout não confirma se a operação aconteceu.

In [ ]:
PAUSA_BASE = 0.5
REPETIVEIS = {'consultar_produto', 'consultar_frete', 'consultar_estoque'}


def com_retry(funcao, *args, **kwargs):
    for tentativa in range(MAX_TENTATIVAS):
        try:
            return funcao(*args, **kwargs)
        except TimeoutError:
            if tentativa == MAX_TENTATIVAS - 1:
                raise
            pausa = PAUSA_BASE * 2 ** tentativa
            print(f'    timeout na tentativa {tentativa + 1}; nova tentativa em {pausa}s')
            time.sleep(pausa)


def falha(tipo, erro, **extra):
    return {'ok': False, 'tipo': tipo, 'erro': erro, **extra}


FERRAMENTAS = {
    'consultar_produto': consultar_produto,
    'consultar_frete': consultar_frete,
    'consultar_estoque': consultar_estoque,
    'calcular_total': calcular_total,
}


def executar_ferramenta(nome, argumentos):
    if nome not in FERRAMENTAS:
        return falha('ferramenta_desconhecida', f'{nome} não existe')
    if not isinstance(argumentos, dict):
        return falha('entrada_invalida', 'argumentos devem ser um objeto JSON')
    funcao = FERRAMENTAS[nome]
    try:
        if nome in REPETIVEIS:
            dados = com_retry(funcao, **argumentos)
        else:
            dados = funcao(**argumentos)
        return {'ok': True, 'dados': dados}
    except TimeoutError as e:
        return falha('timeout', f'{e} (após {MAX_TENTATIVAS} tentativas)')
    except EntradaInvalida as e:
        return falha('entrada_invalida', str(e))
    except NaoEncontrado as e:
        return falha('nao_encontrado', str(e), opcoes=e.opcoes)
    except TypeError as e:
        return falha('entrada_invalida', f'argumentos inválidos: {e}')


print(executar_ferramenta('consultar_produto', {'item': 'Camiseta'}))
print(executar_ferramenta('calcular_total', {'quantidade': 2, 'preco': 35, 'frete': 8}))

## Descrições para o modelo

`consultar_especialista` não está em `FERRAMENTAS`: quem a executa é o próprio laço, porque ela tem limite próprio (uma por pedido).

In [ ]:
def _tool(nome, descricao, propriedades):
    return {'type': 'function', 'function': {
        'name': nome, 'description': descricao,
        'parameters': {'type': 'object', 'properties': propriedades,
                       'required': list(propriedades)}}}


TOOLS = [
    _tool('consultar_produto',
          'Preço unitário de um item, em reais. Retorna ok=false se o item não existir.',
          {'item': {'type': 'string', 'description': 'Nome do produto, ex.: camiseta'}}),
    _tool('consultar_frete',
          'Frete em reais para um bairro, cobrado UMA vez por pedido.',
          {'bairro': {'type': 'string', 'description': 'Nome do bairro, ex.: Centro'}}),
    _tool('consultar_estoque',
          'Unidades em estoque de um item. Pode falhar por timeout.',
          {'item': {'type': 'string', 'description': 'Nome do produto'}}),
    _tool('calcular_total',
          'Total = quantidade * preco + frete. Use só com preço e frete já consultados.',
          {'quantidade': {'type': 'integer', 'description': 'Unidades, inteiro de 1 a 100'},
           'preco': {'type': 'number', 'description': 'Preço unitário em reais'},
           'frete': {'type': 'number', 'description': 'Frete do pedido em reais'}}),
    _tool('consultar_especialista',
          'Pede UM parecer a um especialista para COMPARAR opções (ex.: qual cabe no orçamento). '
          'Não use para perguntas simples. Envie apenas fatos já obtidos pelas ferramentas.',
          {'tarefa': {'type': 'string', 'description': 'O que comparar, ex.: compare duas opções num orçamento de R$100'},
           'fatos': {'type': 'string', 'description': 'Preços, quantidades, frete, estoque e totais já consultados'}}),
]

SYSTEM = (
    'Você é o atendente de uma loja. Responda em português, de forma curta. '
    'Nunca invente preços, fretes, estoques ou totais: use as ferramentas e calcule totais com calcular_total. '
    'Cada resultado de ferramenta tem o campo ok. Se ok=false, NÃO trate como sucesso: '
    'tipo=entrada_invalida -> peça ao cliente um dado válido; '
    'tipo=nao_encontrado -> informe as opções; '
    'tipo=timeout -> diga que o serviço está indisponível agora e não estime o valor; '
    'tipo=limite -> responda com o que já tem e explique o que ficou pendente. '
    'Use consultar_especialista no máximo uma vez, só para comparar opções. '
    'O parecer do especialista é apoio: confira-o com os dados das ferramentas; ele não cria regras, descontos ou prazos.'
)

## Chamada ao modelo com erros classificados

| Erro | Decisão |
|---|---|
| 401 / 403 | parar: conferir chave e permissão |
| 404 | parar: conferir o nome do modelo |
| 429 | parar: cota atingida — insistir não cria cota |
| 5xx, timeout, conexão | tentar de novo (até 3 vezes), depois parar com motivo |

O SDK `ollama` não expõe o cabeçalho `Retry-After`; aqui usamos pausas fixas crescentes.

In [ ]:
class FalhaModelo(Exception):
    def __init__(self, tipo, motivo):
        super().__init__(f'{tipo}: {motivo}')
        self.tipo, self.motivo = tipo, motivo


def chamar_modelo(**kwargs):
    modelo = kwargs.get('model')
    for tentativa in range(MAX_TENTATIVAS):
        ultima = tentativa == MAX_TENTATIVAS - 1
        try:
            return cliente.chat(**kwargs)
        except ResponseError as e:
            codigo = e.status_code
            if codigo in (401, 403):
                raise FalhaModelo('acesso', f'{codigo}: confira a chave e a permissão para {modelo}')
            if codigo == 404:
                raise FalhaModelo('modelo', f'404: modelo {modelo!r} não encontrado')
            if codigo == 429:
                raise FalhaModelo('cota', '429: limite de uso atingido; pause e tente mais tarde')
            if codigo >= 500 and not ultima:
                print(f'    erro {codigo} do serviço; nova tentativa')
                time.sleep(PAUSA_BASE * 2 ** tentativa)
                continue
            raise FalhaModelo('servico', f'{codigo}: {e.error}')
        except (httpx.TimeoutException, httpx.TransportError, ConnectionError) as e:
            if ultima:
                raise FalhaModelo('timeout', f'sem resposta após {MAX_TENTATIVAS} tentativas ({type(e).__name__})')
            print(f'    {type(e).__name__}; nova tentativa')
            time.sleep(PAUSA_BASE * 2 ** tentativa)


def _para_nativo(msg):
    if msg['role'] == 'assistant' and msg.get('tool_calls'):
        return {'role': 'assistant', 'content': msg.get('content') or '',
                'tool_calls': [{'function': {'name': c['name'], 'arguments': c['arguments']}}
                               for c in msg['tool_calls']]}
    if msg['role'] == 'tool':
        return {'role': 'tool', 'tool_name': msg['tool_name'], 'content': msg['content']}
    return {'role': msg['role'], 'content': msg.get('content') or ''}


def pedir_nativo(historico, modelo, tools):
    response = chamar_modelo(model=modelo, messages=[_para_nativo(m) for m in historico],
                             tools=tools, options={'num_predict': 1024})
    m = response.message
    chamadas = [{'id': f'c{i}', 'name': c.function.name, 'arguments': dict(c.function.arguments or {})}
                for i, c in enumerate(m.tool_calls or [])]
    return {'role': 'assistant', 'content': m.content or '', 'tool_calls': chamadas}

## O especialista

Recebe só o necessário: **tarefa**, **fatos** já consultados e **limites**. Não recebe ferramentas nem o histórico inteiro.
Devolve um parecer curto, com condições e dúvidas restantes.

In [ ]:
SYSTEM_ESPECIALISTA = (
    'Você é um especialista em comparar opções de compra. Use SOMENTE os fatos recebidos. '
    'Não invente descontos, prazos, preferências ou preços. Se faltar informação, diga qual. '
    'Responda em até 5 linhas: recomendação condicional, condições e dúvidas restantes.'
)
LIMITES_ESPECIALISTA = 'Sem regra de desconto ou prazo inventada. Não faça novas contas além das necessárias.'


def consultar_especialista(tarefa, fatos):
    mensagens = [
        {'role': 'system', 'content': SYSTEM_ESPECIALISTA},
        {'role': 'user', 'content': f'TAREFA: {tarefa}\nFATOS: {fatos}\nLIMITES: {LIMITES_ESPECIALISTA}'},
    ]
    resposta = chamar_modelo(model=MODELO_ESPECIALISTA, messages=mensagens, options={'num_predict': 1024})
    return {'parecer': resposta.message.content,
            'aviso': 'parecer é apoio à decisão; confira com os resultados das ferramentas'}


def delegar(argumentos, estado):
    if estado['delegacoes'] >= MAX_DELEGACOES:
        return falha('limite', f'limite de {MAX_DELEGACOES} delegação(ões) por pedido já usado')
    if not isinstance(argumentos, dict) or not all(type(argumentos.get(k)) is str for k in ('tarefa', 'fatos')):
        return falha('entrada_invalida', 'envie tarefa e fatos como texto')
    estado['delegacoes'] += 1
    try:
        return {'ok': True, 'dados': consultar_especialista(argumentos['tarefa'], argumentos['fatos'])}
    except FalhaModelo as e:
        return falha('especialista_indisponivel', str(e))

## O laço do coordenador

Guarda o percurso completo: o pedido `assistant` com `tool_calls`, **um resultado `tool` para cada chamada** (inclusive erros) e a resposta final.
Se um lote passa do limite, o lote inteiro é recusado com um motivo.

Parar também é um resultado: `rodar_agente` devolve um **status** (`ok`, `erro_modelo`, `limite_rodadas`) e o motivo.

In [ ]:
def rodar_agente(historico, modelo=None, pedir=None, verbose=True):
    modelo = modelo or MODELO_COORDENADOR
    pedir = pedir or pedir_nativo
    estado = {'rodadas': 0, 'chamadas': 0, 'delegacoes': 0}
    inicio = time.perf_counter()

    def fim(status, resposta='', motivo=''):
        return {'status': status, 'resposta': resposta, 'motivo': motivo, **estado,
                'segundos': round(time.perf_counter() - inicio, 1)}

    for rodada in range(1, MAX_RODADAS + 1):
        estado['rodadas'] = rodada
        try:
            mensagem = pedir(historico, modelo, TOOLS)
        except FalhaModelo as e:
            return fim('erro_modelo', motivo=str(e))
        historico.append(mensagem)

        chamadas = mensagem['tool_calls']
        if not chamadas:
            return fim('ok', resposta=mensagem['content'])

        motivo_lote = None
        if len(chamadas) > MAX_CHAMADAS_POR_RODADA:
            motivo_lote = (f'lote com {len(chamadas)} chamadas recusado: limite de '
                           f'{MAX_CHAMADAS_POR_RODADA} por rodada')
        elif estado['chamadas'] + len(chamadas) > MAX_CHAMADAS_TOTAL:
            motivo_lote = f'lote recusado: limite de {MAX_CHAMADAS_TOTAL} execuções por pedido'

        for c in chamadas:
            if motivo_lote:
                resultado = falha('limite', motivo_lote)
            elif c['name'] == 'consultar_especialista':
                resultado = delegar(c['arguments'], estado)
                estado['chamadas'] += 1
            else:
                resultado = executar_ferramenta(c['name'], c['arguments'])
                estado['chamadas'] += 1
            if verbose:
                marca = 'ok ' if resultado['ok'] else 'ERRO'
                print(f'  [rodada {rodada}] {marca} {c["name"]}({c["arguments"]})')
                if not resultado['ok']:
                    print(f'             -> {resultado["tipo"]}: {resultado["erro"]}')
            historico.append({'role': 'tool', 'tool_call_id': c['id'], 'tool_name': c['name'],
                              'content': json.dumps(resultado, ensure_ascii=False)})

    return fim('limite_rodadas', motivo=f'resposta final pendente após {MAX_RODADAS} rodadas')


def novo_historico(pedido=None):
    historico = [{'role': 'system', 'content': SYSTEM}]
    if pedido:
        historico.append({'role': 'user', 'content': pedido})
    return historico

## O coordenador confere o resultado

Antes de mostrar a resposta, o **código** confere: todo valor em reais citado na resposta precisa aparecer
em algum resultado `ok` de ferramenta (ou no próprio pedido da pessoa, como o orçamento).
O parecer do especialista **não** conta como evidência.

In [ ]:
def _numeros(obj):
    if isinstance(obj, bool):
        return set()
    if isinstance(obj, (int, float)):
        return {round(float(obj), 2)}
    if isinstance(obj, dict):
        return set().union(*map(_numeros, obj.values())) if obj else set()
    if isinstance(obj, list):
        return set().union(*map(_numeros, obj)) if obj else set()
    return set()


def _reais(texto):
    valores = re.findall(r'R\$\s*(\d+(?:[.,]\d{1,2})?)', texto or '')
    return {round(float(v.replace(',', '.')), 2) for v in valores}


def conferir_valores(historico, resposta):
    evidencias = set()
    for m in historico:
        if m['role'] == 'user':
            evidencias |= _reais(m['content'])
        if m['role'] == 'tool' and m['tool_name'] != 'consultar_especialista':
            r = json.loads(m['content'])
            if r.get('ok'):
                evidencias |= _numeros(r['dados'])
    return sorted(_reais(resposta) - evidencias)


def mostrar(resultado, historico=None):
    print(f"\nstatus: {resultado['status']}   rodadas: {resultado['rodadas']}   "
          f"execuções: {resultado['chamadas']}   delegações: {resultado['delegacoes']}   "
          f"tempo: {resultado['segundos']}s")
    if resultado['motivo']:
        print('motivo:', resultado['motivo'])
    if resultado['resposta']:
        print('\nResposta:\n' + resultado['resposta'])
        if historico is not None:
            sem_suporte = conferir_valores(historico, resultado['resposta'])
            print('\nConferência:', 'valores com suporte nas ferramentas' if not sem_suporte
                  else f'ATENÇÃO, valores sem suporte: {sem_suporte}')

Teste rápido com o modelo de verdade (referência: **R$78**):

In [ ]:
historico = novo_historico('Quanto custam 2 camisetas para o Centro?')
resultado = rodar_agente(historico)
mostrar(resultado, historico)

---
# Desafios

## Desafio 1 — Entrada inválida: corrigir, não repetir

Primeiro direto no Python. Repare: nenhum desses erros dispara retry.

In [ ]:
for qtd in (-2, 0, True, '2', 2.5, 101, 3):
    r = executar_ferramenta('calcular_total', {'quantidade': qtd, 'preco': 35, 'frete': 8})
    print(f'{qtd!r:>6} ->', r)

print(executar_ferramenta('calcular_total', {'quantidade': 2}))
print(executar_ferramenta('consultar_produto', {'item': 'bermuda'}))

Agora pelo agente. O esperado é que ele **peça uma quantidade válida** em vez de inventar um total.

In [ ]:
historico = novo_historico('Quanto custam -2 camisetas para o Centro?')
mostrar(rodar_agente(historico), historico)

## Desafio 2 — Falha temporária: retry e parada honesta

- `FALHAS_SIMULADAS = 2`: as duas primeiras tentativas falham, a terceira funciona.
- `FALHAS_SIMULADAS = 3`: as três falham e o erro volta como `{'ok': False, 'tipo': 'timeout'}`.

In [ ]:
FALHAS_SIMULADAS = 2
print('2 falhas:', executar_ferramenta('consultar_estoque', {'item': 'camiseta'}))

FALHAS_SIMULADAS = 3
print('3 falhas:', executar_ferramenta('consultar_estoque', {'item': 'camiseta'}))
FALHAS_SIMULADAS = 0

Com o agente: o estoque fica indisponível. Ele **não deve inventar** o número de unidades.

In [ ]:
FALHAS_SIMULADAS = 3 * 3
historico = novo_historico('Tem 5 camisetas em estoque para entregar no Centro? Quanto fica?')
mostrar(rodar_agente(historico), historico)
FALHAS_SIMULADAS = 0

**Para pensar:** o preço e o total podem aparecer mesmo sem estoque? A resposta deixou claro o que ficou pendente?

## Desafio 3 — Limites de rodadas e de lote

Para ver os limites sem gastar cota, usamos dois "modelos" falsos com o mesmo contrato de `pedir_nativo`:
- `pedir_teimoso` pede ferramenta para sempre → para no limite de rodadas
- `pedir_lote_grande` pede 6 consultas de uma vez → o lote é recusado com motivo, e cada chamada recebe um resultado

In [ ]:
def pedir_teimoso(historico, modelo, tools):
    return {'role': 'assistant', 'content': '', 'tool_calls': [
        {'id': 'x', 'name': 'consultar_produto', 'arguments': {'item': 'camiseta'}}]}


def pedir_lote_grande(historico, modelo, tools):
    if historico[-1]['role'] == 'tool':
        return {'role': 'assistant', 'content': 'Não consegui consultar tudo de uma vez.', 'tool_calls': []}
    return {'role': 'assistant', 'content': '', 'tool_calls': [
        {'id': f'l{i}', 'name': 'consultar_produto', 'arguments': {'item': 'camiseta'}} for i in range(6)]}


h = novo_historico('teste')
mostrar(rodar_agente(h, pedir=pedir_teimoso, verbose=False))

h = novo_historico('teste')
mostrar(rodar_agente(h, pedir=pedir_lote_grande))
print('resultados tool no histórico:', sum(m['role'] == 'tool' for m in h))

Agora com o modelo de verdade e um teto apertado: com `MAX_RODADAS = 1` não dá tempo de consultar **e** responder.

In [ ]:
MAX_RODADAS_ORIGINAL = MAX_RODADAS
MAX_RODADAS = 1
try:
    historico = novo_historico('Quanto custam 2 camisetas para o Centro?')
    mostrar(rodar_agente(historico), historico)
finally:
    MAX_RODADAS = MAX_RODADAS_ORIGINAL

## Desafio 4 — Erros do serviço

Trocamos o cliente temporariamente para provocar cada erro. Repare que **401 e 404 não são repetidos**
e que o timeout é repetido até o limite.

In [ ]:
cliente_original = cliente
modelo_original = MODELO_COORDENADOR

casos = {
    'chave errada': (Client(host=HOST, headers={'Authorization': 'Bearer chave-errada'},
                            timeout=TIMEOUT_REQUISICAO), modelo_original),
    'modelo inexistente': (cliente_original, 'modelo-que-nao-existe:1b'),
    'timeout curtíssimo': (Client(host=HOST, headers=HEADERS, timeout=0.001), modelo_original),
}

try:
    for nome, (cli, modelo) in casos.items():
        cliente = cli
        print(f'\n=== {nome} ===')
        r = rodar_agente(novo_historico('Quanto custa 1 caneca?'), modelo=modelo, verbose=False)
        print(r['status'], '|', r['motivo'], f"| {r['segundos']}s")
finally:
    cliente = cliente_original

**Para pensar:** por que repetir um 429 não resolve? O que mudaria se a operação fosse uma **compra** e não uma consulta?

## Desafio 5 — Especialista: duas camisetas ou três canecas?

Cenário: orçamento de **R$100**, entrega no **Centro**. Preços R$35 e R$22, frete R$8 por pedido.
Totais de referência: **R$78** (2 camisetas) e **R$74** (3 canecas). Preferências e descontos **não** foram informados.

Esperado: o coordenador consulta preços, frete e totais, delega **uma** comparação e confere o parecer.

In [ ]:
print('Referência:', calcular_total(2, 35, 8)['total_reais'], 'e', calcular_total(3, 22, 8)['total_reais'])

PEDIDO_COMPARACAO = ('Tenho R$100. Compro duas camisetas ou três canecas, com entrega no Centro? '
                     'Me ajude a decidir.')
historico = novo_historico(PEDIDO_COMPARACAO)
resultado = rodar_agente(historico)
mostrar(resultado, historico)

O que o especialista **recebeu**? (é isso que a conferência precisa avaliar)

In [ ]:
for m in historico:
    if m['role'] == 'assistant':
        for c in m['tool_calls']:
            if c['name'] == 'consultar_especialista':
                print('ENVIADO AO ESPECIALISTA:\n', json.dumps(c['arguments'], indent=2, ensure_ascii=False))
    if m['role'] == 'tool' and m['tool_name'] == 'consultar_especialista':
        print('\nPARECER:\n', json.loads(m['content']))

**Para pensar:** os fatos enviados estavam corretos e completos? O parecer inventou alguma regra (desconto, prazo)?
A resposta final usou os totais das ferramentas ou números do parecer?

## Desafio 6 — Quando delegar (e quanto custa)

Uma consulta simples deve caber no caminho mais curto. Compare **acerto, tempo e chamadas**:
1. pergunta simples (não deveria delegar)
2. comparação **com** especialista (`MAX_DELEGACOES = 1`)
3. comparação **sem** especialista (`MAX_DELEGACOES = 0`)

In [ ]:
def medir(rotulo, pedido, delegacoes):
    global MAX_DELEGACOES
    original, MAX_DELEGACOES = MAX_DELEGACOES, delegacoes
    try:
        h = novo_historico(pedido)
        r = rodar_agente(h, verbose=False)
    finally:
        MAX_DELEGACOES = original
    sem_suporte = conferir_valores(h, r['resposta']) if r['resposta'] else ['-']
    return {'caso': rotulo, 'status': r['status'], 'rodadas': r['rodadas'], 'execuções': r['chamadas'],
            'delegações': r['delegacoes'], 'segundos': r['segundos'],
            'cita 78': '78' in r['resposta'], 'cita 74': '74' in r['resposta'],
            'valores sem suporte': sem_suporte}


linhas = [
    medir('simples', 'Quanto custam 2 camisetas para o Centro?', 1),
    medir('comparação + especialista', PEDIDO_COMPARACAO, 1),
    medir('comparação sem especialista', PEDIDO_COMPARACAO, 0),
]
for linha in linhas:
    print(linha)

**Para pensar:** a pergunta simples delegou? Se sim, ajuste a descrição de `consultar_especialista` ou o `SYSTEM`.
A versão com especialista foi melhor o bastante para justificar o tempo e as chamadas a mais?

## Desafio 7 — Miniprojeto: agente robusto com delegação

Conversa com `sair` / `novo`. O histórico continua entre perguntas; os limites e a delegação valem **por pedido**.
Teste, nesta ordem:
1. um pedido completo (`2 camisetas no Centro`)
2. um dado ausente (`1 bermuda`)
3. uma entrada inválida (`-3 canecas`)
4. uma comparação (`R$100: camisetas ou canecas?`)
5. digite `falha` para simular o estoque fora do ar e peça o estoque

In [ ]:
historico = novo_historico()
print("Digite 'sair' para encerrar, 'novo' para recomeçar ou 'falha' para derrubar o estoque.")

while True:
    texto = input('\nVocê: ').strip()
    if texto.lower() == 'sair':
        break
    if texto.lower() == 'novo':
        historico = novo_historico()
        print('(histórico reiniciado)')
        continue
    if texto.lower() == 'falha':
        FALHAS_SIMULADAS = 9
        print('(estoque fora do ar nas próximas consultas)')
        continue
    if not texto:
        continue
    historico.append({'role': 'user', 'content': texto})
    resultado = rodar_agente(historico)
    mostrar(resultado, historico)
    if resultado['status'] == 'erro_modelo':
        print('(o pedido não foi concluído; corrija o problema e tente de novo)')

FALHAS_SIMULADAS = 0
print(f'\nFim. O histórico tinha {len(historico)} mensagens.')

---
**O que você consegue explicar agora?**
- Quando **repetir** (timeout em consulta), quando **corrigir** (entrada inválida) e quando **parar** (401/403, 404, 429, limites).
- O que o especialista **recebeu** e o que o coordenador **conferiu**.

**Próxima aula:** ferramentas descobertas por MCP e padrões multiagente.